In [ ]:
### Paquetes
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

### Importar información
from google.colab import drive
drive.mount('/content/drive')

## Carga de Informacion
ruta = "/content/drive/MyDrive/Metodos Cuantitativos y Analisis de Datos/Datos/Rendimientos_REIT.csv"
datos = pd.read_csv(ruta, sep = ";")
print(datos)

### Modelo Parciales e Interpretación Geométrica
## Variables explicativas: Cuantitativa + Categorica
mod_a = smf.ols("r_reit ~ delta_rate + down_market", data=datos).fit()
print(mod_a.summary())

## Gráfico de ajuste (es como sí)
plt.figure(figsize=(6,4))

# Putnos
for g in datos["down_market"].unique():
    sub = datos[datos["down_market"] == g]
    plt.scatter(sub["delta_rate"], sub["r_reit"], alpha=0.7, label=f"down_market={g}")

# Rectas ajustadas
x = np.linspace(datos["delta_rate"].min(), datos["delta_rate"].max(), 100)

for g in datos["down_market"].unique():
    df_pred = pd.DataFrame({
        "delta_rate": x,
        "down_market": g
    })
    y = mod_a.predict(df_pred)
    plt.plot(x, y, linewidth=2)

plt.xlabel("delta_rate")
plt.ylabel("r_reit")
plt.title("Cuantitativa + categórica")
plt.legend()
plt.tight_layout()
plt.show()

## Variables explicativas: Dos cuantitativas
mod_b = smf.ols("r_reit ~ delta_rate + vix_change", data=datos).fit()
print(mod_b.summary())

## Gráfico de ajuste
fig = plt.figure(figsize=(7,5))
ax = fig.add_subplot(111, projection='3d')

# Puntos
ax.scatter(datos["delta_rate"], datos["vix_change"], datos["r_reit"], alpha=0.7)

# Plano ajustado
x = np.linspace(datos["delta_rate"].min(), datos["delta_rate"].max(), 20)
y = np.linspace(datos["vix_change"].min(), datos["vix_change"].max(), 20)
X, Y = np.meshgrid(x, y)

df_pred = pd.DataFrame({
    "delta_rate": X.ravel(),
    "vix_change": Y.ravel()
})
Z = mod_b.predict(df_pred).values.reshape(X.shape)

ax.plot_surface(X, Y, Z, alpha=0.4)

ax.set_xlabel("delta_rate")
ax.set_ylabel("vix_change")
ax.set_zlabel("r_reit")
ax.set_title("Dos cuantitativas")
plt.tight_layout()
plt.show()

### Modelo de Regresion Multiple
mod = smf.ols("r_reit ~ delta_rate + vix_change + cci_change + down_market", data=datos).fit()
print(mod.summary())

from statsmodels.stats.outliers_influence import variance_inflation_factor

### Diagnóstico de multicolinealidad

# Matriz de diseño usada por statsmodels
X = pd.DataFrame(
    mod.model.exog,
    columns=mod.model.exog_names
)

## Factor de Inflación de la Varianza, VIF
vif = pd.DataFrame()
vif["variable"] = X.columns
vif["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

# Usualmente no se interpreta el VIF del intercepto
vif = vif[vif["variable"] != "Intercept"]

print("\nFactores de Inflación de la Varianza, VIF:")
print(vif.sort_values("VIF", ascending=False))

## Índices de condicionamiento

# Excluir intercepto
X_no_intercept = X.drop(columns=["Intercept"], errors="ignore")

# Estandarizar variables
X_std = (X_no_intercept - X_no_intercept.mean()) / X_no_intercept.std(ddof=0)

# Matriz X'X
XtX = np.dot(X_std.T, X_std)

# Autovalores
eigenvalues = np.linalg.eigvals(XtX)
eigenvalues = np.real(eigenvalues)

# Índices de condicionamiento
condition_indices = np.sqrt(eigenvalues.max() / eigenvalues)

cond_df = pd.DataFrame({
    "eigenvalue": eigenvalues,
    "condition_index": condition_indices
}).sort_values("condition_index", ascending=False)

print("\nÍndices de condicionamiento:")
print(cond_df)


### Diagnóstico sobre los residuos
## Preparacion Informacion
pred_df = mod.get_prediction(datos).summary_frame()
infl_df = mod.get_influence().summary_frame()

aug = pd.concat(
    [datos.reset_index(drop=True),
     pred_df[["mean"]].rename(columns={"mean": "fitted"}),
     infl_df[["standard_resid", "student_resid", "hat_diag", "cooks_d"]]],
    axis=1
)
aug["resid"] = aug["r_reit"] - aug["fitted"]
aug["id"]    = np.arange(1, len(aug) + 1)

## Residuos versus Predichos
plt.figure(figsize=(6, 4))
sns.scatterplot(x=aug["fitted"], y=aug["resid"])
plt.axhline(0, ls="--", color="gray")
plt.xlabel("Valores ajustados")
plt.ylabel("Residuos")
plt.title("Residuos vs Ajustados")
plt.show()

## Constraste de Homoscedasticidad
residuos = mod.resid
matriz_exog = mod.model.exog

lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(residuos, matriz_exog)
print("Breusch–Pagan LM statistic:", lm_stat)
print("Breusch–Pagan LM p-value  :", lm_pvalue)

## QQ-PLOT
sm.qqplot(aug["standard_resid"], line="45")
plt.title("QQ-plot residuos estandarizados")
plt.show()

## Tratamiento de las Observaciones
# Leverage (palanca)
print(aug.loc[aug["hat_diag"] > 6/len(aug), ["id", "hat_diag"]].sort_values("hat_diag", ascending=False))
# Influyentes
print(aug.loc[aug["cooks_d"] >= 1, ["id", "cooks_d"]].sort_values("cooks_d", ascending=False))
# Atípicos
print(aug.loc[aug["standard_resid"].abs() > 3, ["id", "standard_resid"]].sort_values("standard_resid", ascending=False))

### Modelo Actualizado Incluyendo Atipico
# Inclusion y Modelo
datos["outlier_221"] = (np.arange(len(datos)) == 220).astype(int)
mod_outlier = smf.ols("r_reit ~ delta_rate + vix_change + cci_change + down_market + outlier_221", data=datos).fit()
print(mod_outlier.summary())

### Diagnóstico sobre los residuos
## Preparacion Informacion
pred_df = mod_outlier.get_prediction(datos).summary_frame()
infl_df = mod_outlier.get_influence().summary_frame()

aug = pd.concat(
    [datos.reset_index(drop=True),
     pred_df[["mean"]].rename(columns={"mean": "fitted"}),
     infl_df[["standard_resid", "student_resid", "hat_diag", "cooks_d"]]],
    axis=1
)
aug["resid"] = aug["r_reit"] - aug["fitted"]
aug["id"]    = np.arange(1, len(aug) + 1)

## Residuos versus Predichos
plt.figure(figsize=(6, 4))
sns.scatterplot(x=aug["fitted"], y=aug["resid"])
plt.axhline(0, ls="--", color="gray")
plt.xlabel("Valores ajustados")
plt.ylabel("Residuos")
plt.title("Residuos vs Ajustados")
plt.show()

## Constraste de Homoscedasticidad
residuos = mod_outlier.resid
matriz_exog = mod_outlier.model.exog

lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(residuos, matriz_exog)
print("Breusch–Pagan LM statistic:", lm_stat)
print("Breusch–Pagan LM p-value  :", lm_pvalue)

## QQ-PLOT
sm.qqplot(aug["standard_resid"], line="45")
plt.title("QQ-plot residuos estandarizados")
plt.show()

## Tratamiento de las Observaciones
# Leverage (palanca)
print(aug.loc[aug["hat_diag"] > 6/len(aug), ["id", "hat_diag"]].sort_values("hat_diag", ascending=False))
# Influyentes
print(aug.loc[aug["cooks_d"] >= 1, ["id", "cooks_d"]].sort_values("cooks_d", ascending=False))
# Atípicos
print(aug.loc[aug["standard_resid"].abs() > 3, ["id", "standard_resid"]].sort_values("standard_resid", ascending=False))


